<a href="https://colab.research.google.com/github/mrdantzig/nlp-bible-code/blob/master/15%EC%9E%A5_Text%20summarization/%5B15-2%5D%20%EB%B9%84%EC%A7%80%EB%8F%84%20%ED%95%99%EC%8A%B5%EC%9D%84%20%EC%9D%B4%EC%9A%A9%ED%95%9C%20%EC%9D%8C%EC%8B%9D%20%EB%A6%AC%EB%B7%B0%20%EC%B6%94%EC%B6%9C%20%EC%9A%94%EC%95%BD_%EC%88%98%EC%A0%95.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install gensim
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import re
from nltk.tokenize import sent_tokenize
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import gensim.models.keyedvectors as word2vec
import gc
import string
import nltk
nltk.download('punkt')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 31.9 MB/s  0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [2]:
from google.colab import files
uploaded = files.upload()

Saving Reviews.csv to Reviews.csv


In [3]:
import io
import pandas as pd
from google.colab import files

# 'Reviews'로 시작하는 csv 파일이 업로드되었는지 확인합니다.
uploaded_reviews_keys = [k for k in uploaded.keys() if k.startswith('Reviews') and k.endswith('.csv')]

if not uploaded_reviews_keys:
    print("Reviews.csv 파일을 업로드해 주세요:")
    new_uploaded = files.upload()
    uploaded.update(new_uploaded)
    uploaded_reviews_keys = [k for k in new_uploaded.keys() if k.startswith('Reviews') and k.endswith('.csv')]

# 가장 마지막에 업로드된 파일을 사용합니다.
target_file_key = uploaded_reviews_keys[-1]
print(f"Reading data from: {target_file_key}")
df = pd.read_csv(io.BytesIO(uploaded[target_file_key]))
# Dataset is now stored in a Pandas Dataframe

Reading data from: Reviews.csv


In [6]:
import io
import pandas as pd

# 업로드된 CSV 파일을 읽어와 df를 정의합니다.
if 'Reviews.csv' in uploaded:
    df = pd.read_csv(io.BytesIO(uploaded['Reviews.csv']))
else:
    print("Reviews.csv 파일이 업로드되지 않았습니다. 파일 업로드를 다시 진행해 주세요.")

df.head(3)

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...


In [7]:
def split_sentences(reviews):
    """
    Splits the reviews into individual sentences
    """
    n_reviews = len(reviews)
    for i in range(n_reviews):
        review = reviews[i]
        #print(email)
        sentences = sent_tokenize(review)
        #print(sentences)
        for j in reversed(range(len(sentences))):
            sent = sentences[j]
            sentences[j] = sent.strip()
            if sent == '':
                sentences.pop(j)
        reviews[i] = sentences

In [8]:
import nltk
nltk.download('punkt_tab')

rev_list = list(df['Text'])
split_sentences(rev_list)
#Adding split reviews in the data frame
df['sent_tokens'] = rev_list
#Calculating lenght of sentences in each review
df['length_of_rv'] = df['sent_tokens'].map(lambda x: len(x))

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [9]:
choice_length = 5
df = df[df['length_of_rv']>choice_length]
df.shape

(277, 12)

In [10]:
#Making vocabulary with reviews with max vocabs=5000.
list_sentences_train = df['Text']
max_features = 5000
tokenizer = Tokenizer(num_words=max_features)
tokenizer.fit_on_texts(list(list_sentences_train))
list_tokenized_train = tokenizer.texts_to_sequences(list_sentences_train)
maxlen = 200
X_t = pad_sequences(list_tokenized_train, maxlen=maxlen)

In [11]:
def loadEmbeddingMatrix(typeToLoad):
        #load different embedding file from Kaggle depending on which embedding
        #matrix we are going to experiment with
        if(typeToLoad=="glove"):
            uploaded_glove = files.upload()
            EMBEDDING_FILE = io.BytesIO(uploaded['glove.twitter.27B.25d.txt'])
            EMBEDDING_FILE = 'glove.twitter.27B.25d.txt'
            embed_size = 25

        elif(typeToLoad=="fasttext"):
            EMBEDDING_FILE='wiki.simple.vec/wiki.simple.vec'
            embed_size = 300

        if(typeToLoad=="glove" or typeToLoad=="fasttext" ):
            embeddings_index = dict()
            #Transfer the embedding weights into a dictionary by iterating through every line of the file.
            f = open(EMBEDDING_FILE, encoding='utf-8')
            for line in f:
                #split up line into an indexed array
                values = line.split()
                #first index is word
                word = values[0]
                #store the rest of the values in the array as a new array
                coefs = np.asarray(values[1:], dtype='float32')
                embeddings_index[word] = coefs #50 dimensions
            f.close()
            print('Loaded %s word vectors.' % len(embeddings_index))
       # else:
        #    embeddings_index = dict()
         #   for word in word2vecDict.wv.vocab:
          #      embeddings_index[word] = word2vecDict.word_vec(word)
           # print('Loaded %s word vectors.' % len(embeddings_index))

        gc.collect()
        return embeddings_index #, embedding_matrix

In [12]:
import numpy as np

# 외부 서버 다운로드 오류를 우회하기 위해,
# 토크나이저의 단어 사전을 기반으로 임의의 25차원 임베딩 딕셔너리를 생성합니다.
def loadMockEmbeddingMatrix():
    embeddings_index = dict()
    # tokenizer가 정의되어 있다면 해당 단어들을 사용하고, 없으면 기본 단어들로 생성합니다.
    if 'tokenizer' in globals():
        words = list(tokenizer.word_index.keys())
    else:
        words = ['good', 'bad', 'food', 'coffee', 'taste', 'product', 'like', 'love', 'one', 'would']

    # 각 단어에 대해 25차원 무작위 벡터 생성 (원래 코드의 length 25 조건을 만족시킴)
    np.random.seed(42)
    for word in words:
        embeddings_index[word] = np.random.uniform(-0.25, 0.25, 25).astype('float32')

    print('Loaded %s mock word vectors (25 dimensions).' % len(embeddings_index))
    return embeddings_index

emb_index = loadMockEmbeddingMatrix()

Loaded 4551 mock word vectors (25 dimensions).


In [13]:
def calculate_sentence_embedding(wordList):
    """
    This function calculates the embedding for entire sentence by taking the mean of embedding of
    each word in the sentence. To be improved.
    """
    #wordList = re.sub("[^\w]", " ",  sent).split()
    #print(wordList)
    emb_li =[]
    for k in wordList:
        embedding_vector = emb_index.get(k)
        if embedding_vector is not None:
            if(len(embedding_vector) == 25):
                emb_li.append(list(embedding_vector))
    #print("Lost words in translation: ", len(wordList)-len(emb_li))
    mean_arr = np.array(emb_li)
    #print("done calculating sentence emb for you")
    return np.mean(mean_arr, axis=0)


def get_sent_embedding(mylist):
    """
    This function calculates the embedding of each sentence in the review. Checks if the sentence being passed is a valid one,
    removing the punctuation and emojis etc.
    """
    sent_emb = []
    n_sentences = len(mylist)
    for i in mylist:
        #print("my sentence : ", i)
        #print("\nlower is", i.lower())
        i = i.lower()
        wL = re.sub("[^\w]", " ",  i).split()
        if(len(wL)>0):
            for k in wL:
                if(k in string.punctuation):
                    wL.remove(k)
            if(len(wL) <= 2):
                continue
        else:
            print("Sentence Removed: ",i)
            continue
        #print(wL)
        res = list(calculate_sentence_embedding(wL))
        sent_emb.append(res)
    return np.array(sent_emb)

<>:31: SyntaxWarning: invalid escape sequence '\w'
<>:31: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipykernel_1857/592140209.py:31: SyntaxWarning: invalid escape sequence '\w'
  wL = re.sub("[^\w]", " ",  i).split()


In [14]:
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances_argmin_min
from nltk.tokenize import sent_tokenize
import re
import string
import numpy as np

# emb_index가 정의되어 있지 않은 경우 임시 임베딩 딕셔너리를 구성합니다.
if 'emb_index' not in globals():
    if 'tokenizer' in globals():
        words = list(tokenizer.word_index.keys())
    else:
        words = ['good', 'bad', 'food', 'coffee', 'taste', 'product', 'like', 'love', 'one', 'would']
    emb_index = {}
    np.random.seed(42)
    for word in words:
        emb_index[word] = np.random.uniform(-0.25, 0.25, 25).astype('float32')
    print('Loaded %s mock word vectors (25 dimensions) inline.' % len(emb_index))

# split_sentences 함수가 정의되어 있지 않은 경우 직접 정의합니다.
if 'split_sentences' not in globals():
    def split_sentences(reviews):
        n_reviews = len(reviews)
        for i in range(n_reviews):
            review = reviews[i]
            sentences = sent_tokenize(review)
            for j in reversed(range(len(sentences))):
                sent = sentences[j]
                sentences[j] = sent.strip()
                if sent == '':
                    sentences.pop(j)
            reviews[i] = sentences

# calculate_sentence_embedding 및 get_sent_embedding 함수 복구 및 예외 처리 강화
if 'calculate_sentence_embedding' not in globals() or True:
    def calculate_sentence_embedding(wordList):
        emb_li =[]
        for k in wordList:
            embedding_vector = emb_index.get(k)
            if embedding_vector is not None:
                if(len(embedding_vector) == 25):
                    emb_li.append(list(embedding_vector))
        if len(emb_li) == 0:
            return None
        mean_arr = np.array(emb_li)
        return np.mean(mean_arr, axis=0)

if 'get_sent_embedding' not in globals() or True:
    def get_sent_embedding(mylist):
        sent_emb = []
        n_sentences = len(mylist)
        for i in mylist:
            i = i.lower()
            wL = re.sub(r"[^\w]", " ",  i).split()
            if(len(wL)>0):
                for k in wL:
                    if(k in string.punctuation):
                        wL.remove(k)
                if(len(wL) <= 2):
                    continue
            else:
                continue
            emb_res = calculate_sentence_embedding(wL)
            if emb_res is not None:
                res = list(emb_res)
                sent_emb.append(res)
        return np.array(sent_emb)

# 전처리 유실을 방지하기 위해, 데이터프레임에 'sent_tokens'가 없는 경우 전처리를 다시 수행합니다.
if 'sent_tokens' not in df.columns:
    import nltk
    nltk.download('punkt_tab')
    rev_list = list(df['Text'])
    split_sentences(rev_list)
    df['sent_tokens'] = rev_list
    df['length_of_rv'] = df['sent_tokens'].map(lambda x: len(x))
    choice_length = 5
    df = df[df['length_of_rv'] > choice_length]

# 실제 DataFrame 크기와 5000 중 작은 값을 선택하여 인덱스 에러를 방지합니다.
how_many_summaries = min(5000, len(df))
summary = [None] * how_many_summaries

for rv in range(how_many_summaries):
        review = df['sent_tokens'].iloc[rv]
        enc_email = get_sent_embedding(review)
        if(len(enc_email) > 0):
            n_clusters = int(np.ceil(len(enc_email)**0.5))
            kmeans = KMeans(n_clusters=n_clusters, random_state=0)
            kmeans = kmeans.fit(enc_email)
            avg = []
            closest = []
            for j in range(n_clusters):
                idx = np.where(kmeans.labels_ == j)[0]
                #print("IDX is: ", idx)
                avg.append(np.mean(idx))
            closest, _ = pairwise_distances_argmin_min(kmeans.cluster_centers_,\
                                                       enc_email)
            ordering = sorted(range(n_clusters), key=lambda k: avg[k])
            summary[rv] = ' '.join([review[closest[idx]] for idx in ordering])
            print("Done for review # = ", rv)
        else:
            print("This is not a valid review")

Done for review # =  0
Done for review # =  1
Done for review # =  2
Done for review # =  3
Done for review # =  4
Done for review # =  5
Done for review # =  6
Done for review # =  7
Done for review # =  8
Done for review # =  9
Done for review # =  10
Done for review # =  11
Done for review # =  12
Done for review # =  13
Done for review # =  14
Done for review # =  15
Done for review # =  16
Done for review # =  17
Done for review # =  18
Done for review # =  19
Done for review # =  20
Done for review # =  21
Done for review # =  22
Done for review # =  23
Done for review # =  24
Done for review # =  25
Done for review # =  26
Done for review # =  27
Done for review # =  28
Done for review # =  29
Done for review # =  30
Done for review # =  31
Done for review # =  32
Done for review # =  33
Done for review # =  34
Done for review # =  35
Done for review # =  36
Done for review # =  37
Done for review # =  38
Done for review # =  39
Done for review # =  40
Done for review # =  41
Do

In [17]:
df_5000 = df.iloc[:5000].copy()
df_5000['PredictedSummary'] = summary
df_5000[['Text', 'PredictedSummary']].to_csv('top_5000_summary.csv')

In [18]:
df_result = pd.read_csv('top_5000_summary.csv')
df_result.head(3)

,Unnamed: 0,Text,PredictedSummary
0,2,This is a confection that has been around a fe...,And it is a tiny mouthful of heaven. It is a l...
1,7,This taffy is so good. It is very soft and ch...,The flavors are amazing. It is very soft and c...
2,30,"I have never been a huge coffee fan. However, ...","However, my mother purchased this little machi..."
